In [1]:
import torch
import torch.nn as nn

# Simple Feedforward Neural Network in PyTorch

This notebook defines and trains a fully connected neural network using `torch.nn.Module`. It takes 5 input features, passes them through a hidden layer with 10 neurons and ReLU activation, then produces scores for 3 output classes. Softmax converts the scores to class probabilities.

## Architecture and parameter count

| Layer | Shape | Trainable parameters |
|---|---:|---:|
| Input → hidden (`Linear(5, 10)`) | 5 inputs → 10 neurons | 50 weights + 10 biases = 60 |
| Hidden → output (`Linear(10, 3)`) | 10 neurons → 3 classes | 30 weights + 3 biases = 33 |
| **Total** | | **80 weights + 13 biases = 93 parameters** |

ReLU and Softmax have no trainable parameters. The notebook uses 20 samples, SGD with a learning rate of 0.1, and 25 training epochs.

**Note:** The training loop generates new random target classes each epoch, so the labels are not consistent. Also, `CrossEntropyLoss` is normally used with raw logits, without a Softmax layer in the model. As written, the training is primarily a demonstration rather than meaningful learning.

In [2]:
class SimpleModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()

        self.linear1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_size, output_size)
        self.softmax = nn.Softmax(dim=1)

    # use sequential container to define the forward pass
    def forward(self, features):
        seq = nn.Sequential(
            self.linear1,
            self.relu,
            self.linear2,
            self.softmax
        )

        output = seq(features)

        return output


Make a random dataset of 5 features

In [24]:
input_tensor = torch.rand(20, 5) # input tensor with 10 samples and 5 features
input_tensor

tensor([[0.8226, 0.2078, 0.6553, 0.8040, 0.4227],
        [0.9587, 0.9135, 0.3197, 0.0589, 0.7648],
        [0.5754, 0.4638, 0.3992, 0.6298, 0.8630],
        [0.6597, 0.8127, 0.9949, 0.3443, 0.3653],
        [0.0672, 0.2128, 0.1236, 0.5194, 0.7820],
        [0.3379, 0.5746, 0.4407, 0.3034, 0.6964],
        [0.0646, 0.3274, 0.6948, 0.0553, 0.6486],
        [0.7767, 0.1290, 0.4682, 0.1062, 0.0838],
        [0.6588, 0.4048, 0.4122, 0.8117, 0.5408],
        [0.8069, 0.9594, 0.9047, 0.0467, 0.0153],
        [0.4142, 0.6699, 0.1418, 0.6381, 0.8482],
        [0.0261, 0.2083, 0.1731, 0.3268, 0.4757],
        [0.7282, 0.3227, 0.4556, 0.9018, 0.1554],
        [0.5607, 0.1890, 0.2639, 0.7945, 0.1839],
        [0.2869, 0.3214, 0.5486, 0.7316, 0.0468],
        [0.7790, 0.2027, 0.1390, 0.1059, 0.2596],
        [0.8469, 0.6191, 0.2910, 0.8544, 0.9738],
        [0.5547, 0.5691, 0.5201, 0.3766, 0.6104],
        [0.5717, 0.5608, 0.4200, 0.0930, 0.5240],
        [0.9571, 0.0673, 0.2114, 0.2603, 0.2541]])

In [42]:
# define parameters for the model
input_size = 5
hidden_size = 10
output_size = 3
learning_rate = 0.1
epochs = 25
loss_function = nn.CrossEntropyLoss()


Make the model

In [43]:
# create an instance of the model
model = SimpleModel(input_size, hidden_size, output_size)
# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define a funtion to train the model
def train_model(model, optimizer, loss_function, input_tensor, epochs):
    for epoch in range(epochs):

        # forward pass
        output = model(input_tensor)
        # compute loss
        loss = loss_function(output, torch.randint(0, output_size, (input_tensor.size(0),)))

        # backward pass

        # clear gradients
        optimizer.zero_grad()
        # compute gradients
        loss.backward()
        # update parameters
        optimizer.step()

        # print loss in each epoch
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")


        

Train The Model

In [44]:
# start training the model
train_model(model, optimizer, loss_function, input_tensor, epochs)

Epoch [1/25], Loss: 1.1116
Epoch [2/25], Loss: 1.0913
Epoch [3/25], Loss: 1.0852
Epoch [4/25], Loss: 1.1089
Epoch [5/25], Loss: 1.1136
Epoch [6/25], Loss: 1.1133
Epoch [7/25], Loss: 1.0828
Epoch [8/25], Loss: 1.1104
Epoch [9/25], Loss: 1.0968
Epoch [10/25], Loss: 1.1100
Epoch [11/25], Loss: 1.1020
Epoch [12/25], Loss: 1.0944
Epoch [13/25], Loss: 1.1079
Epoch [14/25], Loss: 1.0880
Epoch [15/25], Loss: 1.1043
Epoch [16/25], Loss: 1.0996
Epoch [17/25], Loss: 1.0980
Epoch [18/25], Loss: 1.1032
Epoch [19/25], Loss: 1.1046
Epoch [20/25], Loss: 1.0860
Epoch [21/25], Loss: 1.1122
Epoch [22/25], Loss: 1.0948
Epoch [23/25], Loss: 1.0915
Epoch [24/25], Loss: 1.1113
Epoch [25/25], Loss: 1.0906


In [45]:
# see all model weights
weights = model.state_dict()
weights

OrderedDict([('linear1.weight',
              tensor([[-0.1196,  0.0835,  0.2117, -0.3766,  0.4120],
                      [-0.4255, -0.0849, -0.2347,  0.1862,  0.3057],
                      [ 0.1334,  0.2301,  0.3090,  0.3718, -0.1577],
                      [ 0.4012, -0.3188, -0.1985, -0.4003, -0.0594],
                      [-0.2287,  0.0281, -0.1398,  0.3543, -0.3171],
                      [-0.4112, -0.3314, -0.2380, -0.1072,  0.2827],
                      [ 0.0476,  0.1566,  0.2723, -0.3732, -0.1628],
                      [ 0.3165,  0.1400, -0.0425,  0.0623,  0.0142],
                      [ 0.0055,  0.3329, -0.3725, -0.3352,  0.3401],
                      [ 0.3780,  0.4116,  0.0828,  0.2533, -0.1798]])),
             ('linear1.bias',
              tensor([ 0.1259, -0.3820, -0.4276,  0.0332, -0.2714, -0.4272, -0.3175,  0.2425,
                      -0.2314, -0.4289])),
             ('linear2.weight',
              tensor([[ 0.0381, -0.0770, -0.0930,  0.2392,  0.1911,  0.1823,